# Tiles overview: several SEM tiles at their mask positions

Places a few tiles at their mask positions (tile centre from the metadata CSV) to see how
they overlap:
- each SEM image, semi-transparent. Where two images overlap, a misalignment shows up as
  doubled ("ghost") patterns;
- each tile's outline, plus its `.oas` polygons in the same colour;
- each tile's detected SEM contact centres (`+`) in the same colour;
- a table of how much each pair of tiles overlaps, from the metadata.

The plot is interactive: use the zoom/pan buttons to look at an overlap strip closely.
If no interactive plot appears, change `%matplotlib widget` to `%matplotlib inline`.

In [ ]:
DATA_DIR = r"path\to\data"   # folder with the .jpg files, the metadata .csv and Contour/
TILES = [0, 1, 2, 3, 4]       # which tiles to show (indices in read_tile_index order)
LAYER = (1, 0)                # (layer, datatype) of the contacts in the .oas

In [ ]:
%matplotlib widget
import itertools

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.collections import PolyCollection

from affine_ransac.features.contact import detect_contacts
from affine_ransac.geometry.frames import pixel_to_tile_nm
from affine_ransac.io.design import load_layout, read_polygons
from affine_ransac.io.metadata import read_tile_index
from affine_ransac.io.sem_image import load_sem_image

## Overlap between tiles (from the metadata)

In [ ]:
all_tiles = read_tile_index(DATA_DIR)
tiles = [all_tiles[i] for i in TILES]

rows = []
for a, b in itertools.combinations(tiles, 2):
    overlap_x = (a.fov_x_nm + b.fov_x_nm) / 2 - abs(b.center_x_nm - a.center_x_nm)
    overlap_y = (a.fov_y_nm + b.fov_y_nm) / 2 - abs(b.center_y_nm - a.center_y_nm)
    if overlap_x > 0 and overlap_y > 0:
        rows.append({
            "tile_a": a.tile_id, "tile_b": b.tile_id,
            "step_x_nm": b.center_x_nm - a.center_x_nm, "step_y_nm": b.center_y_nm - a.center_y_nm,
            "overlap_x_nm": overlap_x, "overlap_y_nm": overlap_y,
        })

print(f"{len(rows)} overlapping pairs among {len(tiles)} tiles")
pd.DataFrame(rows)

## All tiles at their mask positions

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
colors = plt.cm.tab10.colors

for k, tile in enumerate(tiles):
    color = colors[k % len(colors)]
    cx, cy = tile.center_x_nm, tile.center_y_nm
    half_w, half_h = tile.fov_x_nm / 2, tile.fov_y_nm / 2

    # SEM image: row 0 at the top (imshow's default), placed by its edges in mask nm.
    image = load_sem_image(tile.image_path)
    ax.imshow(image, cmap="gray", alpha=0.5, extent=(cx - half_w, cx + half_w, cy - half_h, cy + half_h))
    ax.add_patch(plt.Rectangle((cx - half_w, cy - half_h), 2 * half_w, 2 * half_h, fill=False, color=color))
    ax.text(cx, cy, tile.tile_id, color=color, ha="center", va="center", fontsize=12)

    # Design polygons (tile-local, centred on (0, 0)) shifted to mask coordinates.
    if tile.oas_path is not None:
        polygons = [p + (cx, cy) for p in read_polygons(load_layout(tile.oas_path), *LAYER)]
        ax.add_collection(PolyCollection(polygons, facecolors="none", edgecolors=color, linewidths=0.5))

    # SEM contact centres in mask coordinates.
    found = detect_contacts(image)
    sem = pixel_to_tile_nm(found.centers, image.shape, tile.fov_x_nm / image.shape[1]) + (cx, cy)
    ax.scatter(sem[:, 0], sem[:, 1], marker="+", s=20, color=color)

ax.set_xlim(min(t.center_x_nm - t.fov_x_nm / 2 for t in tiles), max(t.center_x_nm + t.fov_x_nm / 2 for t in tiles))
ax.set_ylim(min(t.center_y_nm - t.fov_y_nm / 2 for t in tiles), max(t.center_y_nm + t.fov_y_nm / 2 for t in tiles))
ax.set_aspect("equal")
ax.set_xlabel("mask x (nm)")
ax.set_ylabel("mask y (nm)")
plt.show()